# 01 - Auditimi i datasetit

Ky notebook përdoret vetëm për eksplorim dhe raportim. Ai lexon datasetin e përpunuar nga `data/processed/articles.csv` dhe shfaq kontrollet bazë të Ditës 1.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

sys.path.append(str(PROJECT_ROOT))

from src.data.validate_dataset import validate_dataset

processed_path = PROJECT_ROOT / "data" / "processed" / "articles.csv"
df = pd.read_csv(processed_path, encoding="utf-8-sig")
df.head()

## Shpërndarja e labels

In [ ]:
label_counts = (
    df["label_name"]
    .value_counts()
    .rename_axis("label_name")
    .reset_index(name="count")
)
display(label_counts)

ax = label_counts.plot(kind="bar", x="label_name", y="count", legend=False, color=["#4c78a8", "#f58518"])
ax.set_title("Shpërndarja e artikujve sipas label")
ax.set_xlabel("Label")
ax.set_ylabel("Numri i artikujve")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Missing values për title/content

In [ ]:
missing_summary = pd.DataFrame(
    {
        "field": ["title", "content", "raw_text", "pair_id"],
        "missing_or_empty": [
            df["title"].fillna("").astype(str).str.strip().eq("").sum(),
            df["content"].fillna("").astype(str).str.strip().eq("").sum(),
            df["raw_text"].fillna("").astype(str).str.strip().eq("").sum(),
            df["pair_id"].isna().sum(),
        ],
    }
)
missing_summary

## Shpërndarjet e gjatësisë së tekstit

In [ ]:
lengths = df.assign(
    title_chars=df["title"].fillna("").astype(str).str.len(),
    content_chars=df["content"].fillna("").astype(str).str.len(),
    raw_text_chars=df["raw_text"].fillna("").astype(str).str.len(),
    raw_text_words=df["raw_text"].fillna("").astype(str).str.split().str.len(),
)

display(lengths[["title_chars", "content_chars", "raw_text_chars", "raw_text_words"]].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
lengths.hist(column="raw_text_chars", by="label_name", bins=40, ax=axes, sharex=True, color="#54a24b")
fig.suptitle("Gjatësia e raw_text sipas label")
for ax in axes:
    ax.set_xlabel("Karaktere")
    ax.set_ylabel("Artikuj")
plt.tight_layout()
plt.show()

## Shembuj artikujsh

In [ ]:
for label_name in ["real", "fake"]:
    row = df.loc[df["label_name"] == label_name].sort_values("pair_id").iloc[0]
    print("=" * 80)
    print(f"Label: {label_name} | article_id: {row['article_id']} | pair_id: {row['pair_id']}")
    print(f"Path: {row['file_path']}")
    print("\nTitulli:")
    print(row["title"])
    print("\nPërmbajtja, 1000 karakteret e para:")
    print(row["content"][:1000])


## Përmbledhje e problemeve të datasetit

In [ ]:
summary = validate_dataset(df, print_report=True)

problem_summary = pd.DataFrame(
    [
        {"check": "Missing titles", "value": summary["missing_titles"]},
        {"check": "Missing contents", "value": summary["missing_contents"]},
        {"check": "Missing pair IDs", "value": summary["missing_pair_ids"]},
        {"check": "Duplicate raw text rows", "value": summary["duplicate_raw_text_rows"]},
        {"check": "Duplicate raw text groups", "value": summary["duplicate_raw_text_groups"]},
        {"check": "Short articles", "value": summary["short_articles"]},
        {"check": "Pair IDs only true", "value": summary["pair_ids_only_true"]},
        {"check": "Pair IDs only fake", "value": summary["pair_ids_only_fake"]},
    ]
)
problem_summary